# Mapas do delta de tempo — Curitiba

Compara o tempo de acesso de cada cenário com o cenário típico. Valores positivos indicam aumento do tempo de acesso; valores negativos indicam redução.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

BASE_DIR = Path.cwd().parents[2]
RESULTS_DIR = BASE_DIR / 'results' / 'curitiba'
SCENARIOS = ['lst', 'lst_green_composite', 'lst_demo_k05', 'lst_green_composite_demo_k05']
LABELS = {
    'lst': 'LST (calibrado)',
    'lst_green_composite': 'LST + vegetação (calibrado)',
    'lst_demo_k05': 'LST (demonstrativo, k=0,05)',
    'lst_green_composite_demo_k05': 'LST + vegetação (demonstrativo)',
}
grid = gpd.read_parquet(RESULTS_DIR / 'prepared' / 'grid.parquet')
def access_time(scenario):
    matrix = pd.read_parquet(RESULTS_DIR / 'matrices' / f'matrix_{scenario}.parquet')
    return matrix.groupby('cell_idx')['travel_time'].mean().rename(scenario)
typical = access_time('typical')
deltas = {scenario: access_time(scenario).subtract(typical) for scenario in SCENARIOS}
print('células:', len(grid))
for scenario, delta in deltas.items():
    print(f'{scenario:35s} mediana={delta.median():.2f} min | válidos={delta.notna().sum()}')


## Escala comum

A mesma escala é usada em todos os mapas. O branco representa delta próximo de zero; azul indica redução e vermelho indica aumento do tempo de acesso.

In [ ]:
LIMITE = max(abs(pd.concat(deltas.values()).dropna().quantile([0.02, 0.98])))
LIMITE = max(1.0, float(np.ceil(LIMITE)))
FAIXAS = [-LIMITE, -10, -5, -2, 2, 5, 10, LIMITE]
CORES = ['#2166ac', '#67a9cf', '#d1e5f0', '#f7f7f7', '#fddbc7', '#ef8a62', '#b2182b']
ROTULOS = [f'< {-10} min', '-10 a -5 min', '-5 a -2 min', '-2 a 2 min', '2 a 5 min', '5 a 10 min', '> 10 min']
CMAP = ListedColormap(CORES)
NORM = BoundaryNorm(FAIXAS, len(CORES))
print(f'escala: {FAIXAS[0]:.0f} a {FAIXAS[-1]:.0f} minutos')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 18))
for ax, scenario in zip(axes.flat, SCENARIOS):
    mapa = grid.merge(deltas[scenario].rename('delta_t'), on='cell_idx', how='left')
    mapa.plot(column='delta_t', cmap=CMAP, norm=NORM, ax=ax, edgecolor='none',
              missing_kwds={'color': '#bdbdbd'}, legend=False)
    ax.set_title(LABELS[scenario], fontsize=16, fontweight='bold')
    ax.set_axis_off()
fig.legend(handles=[Patch(facecolor=c, edgecolor='#555', label=l) for c, l in zip(CORES, ROTULOS)],
           loc='lower center', ncol=4, frameon=False, fontsize=12,
           title='Delta do tempo de acesso em relação ao cenário típico (min)',
           bbox_to_anchor=(0.5, 0.01))
fig.suptitle('Variação do tempo de acesso por cenário — Curitiba', fontsize=20, fontweight='bold')
fig.subplots_adjust(top=0.93, bottom=0.10, hspace=0.08, wspace=0.04)
plt.show()


## Interpretação

A leitura deve se concentrar na distribuição espacial dos aumentos e reduções, e não apenas na mediana global. O mapa mostra a variação do indicador de acessibilidade por célula e permite identificar territórios mais sensíveis a cada cenário.